# all

In [1]:
from common import *

DATA = Path("../data")
engine = get_engine()

In [3]:
with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT DB_NAME()")).scalar())
    print("Login:", conn.execute(text("SELECT SUSER_NAME()")).scalar())
    print(pd.read_sql(
        "SELECT name FROM sys.schemas "
        "WHERE name IN ('bronze', 'silver', 'gold')",
        conn
    ))

Database: Datacube
Login: datacube_analytics
     name
0  bronze
1    gold
2  silver


In [36]:
for table, file in [("airlines", "Flight/airlines.csv"), ("airports", "Flight/airports.csv")]:
    pd.read_csv(DATA / file).to_sql(table, engine, schema="bronze", if_exists="replace", index=False)
    print(table, "yükləndi")

airlines yükləndi
airports yükləndi


In [4]:
def load_parquet(table, path, columns=None, dtype=None):
    """Stream a parquet file into bronze.<table>; the first batch replaces the table."""
    for i, batch in enumerate(pq.ParquetFile(path).iter_batches(batch_size=100_000, columns=columns)):
        batch.to_pandas().to_sql(
            table, engine, schema="bronze",
            if_exists="replace" if i == 0 else "append",
            index=False, chunksize=5_000, dtype=dtype,
        )
        print(table, "batch", i, flush=True)

In [ ]:
FLIGHT_COLS = [
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE",
    "SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",
    "TAXI_OUT", "TAXI_IN", "AIR_TIME",
]
FLIGHT_TYPES = {
    "AIRLINE": NVARCHAR(10), "ORIGIN_AIRPORT": NVARCHAR(10),
    "DESTINATION_AIRPORT": NVARCHAR(10), "CANCELLATION_REASON": NVARCHAR(5),
}

load_parquet("flights", DATA / "Flight/flights.parquet", FLIGHT_COLS, FLIGHT_TYPES)

In [38]:
from sqlalchemy import text

with engine.connect() as conn:
    total = conn.execute(
        text("SELECT COUNT_BIG(*) FROM bronze.flights")
    ).scalar_one()

print(f"Yekun sətir sayı: {total:,}")

Yekun sətir sayı: 5,700,000


In [5]:
load_parquet("weather", DATA / "api/weather_2015_all.parquet")

weather batch 0
weather batch 1
weather batch 2
weather batch 3
weather batch 4
weather batch 5
weather batch 6
weather batch 7
weather batch 8
weather batch 9
weather batch 10
weather batch 11
weather batch 12
weather batch 13
weather batch 14
weather batch 15
weather batch 16
weather batch 17
weather batch 18
weather batch 19
weather batch 20
weather batch 21
weather batch 22


In [6]:
for table in ["airlines", "airports", "flights", "weather"]:
    count = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM bronze.{table}"), engine)["n"][0]
    print(table, f"{count:,}")

airlines 14
airports 322


flights 5,700,000
weather 2,225,040


In [16]:
print(pd.read_sql(text("SELECT TOP 5 * FROM bronze.weather"), engine).T)
print(pd.read_sql(text(
    "SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS "
    "WHERE TABLE_SCHEMA='bronze' AND TABLE_NAME='weather'"), engine))

                                     0                    1  \
airport_code                       BFL                  BFL   
observation_time   2015-09-03 09:00:00  2015-09-03 10:00:00   
Latitude                       35.4336              35.4336   
Longitude                   -119.05677           -119.05677   
temperature                       22.0                 20.9   
precipitation                      0.0                  0.0   
wind_speed                         6.4                  7.1   
visibility                        None                 None   
weather_condition                    0                    0   
surface_pressure                 993.7                993.7   
wind_direction                     333                  315   
snowfall                           0.0                  0.0   
wind_gusts                        15.1                 12.6   

                                     2                    3  \
airport_code                       BFL                